# 3 — Borderline Cleaning

Four deep-learning methods for finding NonSEP training samples that closely
resemble SEP events, and removing them. Each method saves its own cleaned
dataset; notebook 4 then trains a GRU on all four to see which improves TSS.

**Input:** `./final_split_data_HybridNorm_BestFeatures` (notebooks 1 and 2).

**Output:** one dataset per method, `{train,val,test}_set.pkl`. Only the
training set is modified; val and test are copied through unchanged.

| method | output folder |
|---|---|
| Deep ENN | `..._DeepENN/` |
| Deep Tomek Links | `..._DeepTomek/` |
| Autoencoder (SEP only) | `..._DeepAE/` |
| Siamese/Triplet | `..._Siamese/` |

## Methods

1. **Deep ENN** — a GRU trained by 5-fold cross-validation gives every NonSEP
   sample an out-of-fold predicted probability. Samples it confidently scores
   as SEP-like are removed. Deep-learning analogue of Edited Nearest
   Neighbours, using classifier confidence in place of a k-NN vote.

2. **Deep Tomek Links** — the same out-of-fold pass yields a 32-d hidden-layer
   embedding per sample. Mutual nearest-neighbour pairs across classes in that
   space are Deep Tomek Links; the NonSEP member of each pair is removed.

3. **Autoencoder novelty detection** — a small autoencoder is trained on the
   SEP samples only. A NonSEP sample it reconstructs as well as a typical SEP
   sample lies on the SEP manifold and is removed. Density-based rather than
   discriminative, and fitted on very few positives, so its threshold is the
   least stable of the four.

4. **Siamese/Triplet network** — an embedding trained explicitly for similarity
   with triplet loss, out-of-fold via 5-fold cross-validation. NonSEP samples
   still close to an SEP sample in this space, despite training that pushes the
   classes apart, are the most persistently ambiguous.

**Run after:** notebook 2. **Run before:** notebook 4.


In [24]:
# ══════════════════════════════════════════════════════════════
# Must run before importing torch/sklearn -- and before any other cell
# in this notebook that does. Mixing PyTorch (CPU or MPS) with
# scikit-learn's threaded neighbor search (used by NearestNeighbors in
# Methods 2 and 4) in the same process segfaults on this machine:
#
#   OMP: Error #179: Function pthread_mutex_init failed:
#   OMP: System error #22: Invalid argument
#
# Confirmed via a minimal repro (GRU forward/backward, then
# NearestNeighbors.fit/.kneighbors -> hard crash, exit code 139) --
# device (CPU vs MPS) made no difference, so this is a native
# OpenMP-runtime conflict between the two libraries, not an MPS issue.
# These env vars force everything onto a single, non-conflicting OpenMP
# runtime and disable the threading that triggers it. No effect on
# results -- sklearn just runs single-threaded.
# ══════════════════════════════════════════════════════════════
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"

## 1. Load Best-Feature Data

In [25]:
import pickle
import numpy as np
import os

def load_split(path):
    with open(path, "rb") as f:
        data = pickle.load(f)
    return data["X"].astype(np.float32), data["y"]

folder = "./final_split_data_HybridNorm_BestFeatures"

X_train, y_train = load_split(f"{folder}/train_set.pkl")
X_val,   y_val   = load_split(f"{folder}/val_set.pkl")
X_test,  y_test  = load_split(f"{folder}/test_set.pkl")

selected_features_path = os.path.join(folder, "selected_features.txt")
if os.path.exists(selected_features_path):
    with open(selected_features_path) as f:
        selected_features = f.readline().strip().split(",")
else:
    selected_features = [f"feature_{i}" for i in range(X_train.shape[2])]

train_counts = np.bincount(y_train.astype(int), minlength=2)
val_counts   = np.bincount(y_val.astype(int),   minlength=2)
test_counts  = np.bincount(y_test.astype(int),  minlength=2)

def save_bundle(X, y, path):
    with open(path, "wb") as f:
        pickle.dump({"X": X, "y": y}, f)
    print(f"Saved: {path}")

print(f"Train: {X_train.shape} | Non-SEP={train_counts[0]} | SEP={train_counts[1]}")
print(f"Val  : {X_val.shape} | Non-SEP={val_counts[0]} | SEP={val_counts[1]}")
print(f"Test : {X_test.shape} | Non-SEP={test_counts[0]} | SEP={test_counts[1]}")
print(f"Features ({len(selected_features)}): {selected_features}")

Train: (12455, 288, 3) | Non-SEP=12337 | SEP=118
Val  : (1780, 288, 3) | Non-SEP=1763 | SEP=17
Test : (3559, 288, 3) | Non-SEP=3525 | SEP=34
Features (3): ['P4', 'Xl', 'Vx']


## 2. Shared Out-of-Fold GRU (Methods 1 & 2)

Trains one GRU classifier via 5-fold stratified cross-validation and
records, for every training sample, both its out-of-fold predicted
probability and its out-of-fold 32-d embedding — reused by Deep ENN
(probabilities) and Deep Tomek Links (embeddings) below, so the model is
only trained once.

Fixed epoch budget, no early stopping: this model only needs to be
discriminative enough to be useful for cleaning, not a final production
classifier (that's notebook 4's job).

In [7]:
import torch
import torch.nn as nn
import time
from sklearn.model_selection import StratifiedKFold

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")


class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=1, dropout=0.1):
        super().__init__()
        self.gru      = nn.GRU(input_size, hidden_size, num_layers=num_layers,
                               batch_first=True,
                               dropout=dropout if num_layers > 1 else 0)
        self.bn       = nn.BatchNorm1d(hidden_size)
        self.dropout  = nn.Dropout(dropout)
        self.fc1      = nn.Linear(hidden_size, 32)
        self.relu     = nn.ReLU()
        self.dropout2 = nn.Dropout(0.2)
        self.fc2      = nn.Linear(32, 1)

    def embed(self, x):
        out, _ = self.gru(x)
        out    = out[:, -1, :]
        out    = self.bn(out)
        out    = self.dropout(out)
        out    = self.relu(self.fc1(out))
        return out

    def forward(self, x):
        out = self.embed(x)
        out = self.dropout2(out)
        return self.fc2(out).squeeze()


N_FOLDS      = 5
EMBED_EPOCHS = 30
EMBED_LOG_EVERY = 5   # print progress every N epochs, plus first/last
embed_params = {"units": 64, "layers": 1, "dropout": 0.1, "lr": 0.001, "batch_size": 64}

class_ratio = train_counts[0] / train_counts[1]

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
oof_probs      = np.zeros(len(y_train), dtype=np.float32)
oof_embeddings = np.zeros((len(y_train), 32), dtype=np.float32)

print("═" * 70)
print("  Shared out-of-fold GRU -- probabilities + embeddings")
print(f"  Folds       : {N_FOLDS}")
print(f"  Params      : {embed_params}")
print(f"  Epochs/fold : {EMBED_EPOCHS} (fixed, no early stopping)")
print("═" * 70)

fold_times = []

for fold, (tr_idx, ho_idx) in enumerate(skf.split(X_train, y_train), start=1):

    fold_t0 = time.time()

    X_tr_fold = torch.tensor(X_train[tr_idx], dtype=torch.float32).to(device)
    y_tr_fold = torch.tensor(y_train[tr_idx], dtype=torch.float32).to(device)
    X_ho_fold = torch.tensor(X_train[ho_idx], dtype=torch.float32).to(device)

    model = GRUModel(X_train.shape[2],
                      hidden_size=embed_params["units"],
                      num_layers=embed_params["layers"],
                      dropout=embed_params["dropout"]).to(device)

    pos_weight = torch.tensor([float(class_ratio)], dtype=torch.float32).to(device)
    criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer  = torch.optim.Adam(model.parameters(), lr=embed_params["lr"])

    n_samples  = X_tr_fold.shape[0]
    batch_size = embed_params["batch_size"]

    print(f"\n  Fold {fold}/{N_FOLDS} -- training on {n_samples} samples "
          f"({int(y_tr_fold.sum().item())} SEP, {n_samples - int(y_tr_fold.sum().item())} Non-SEP)...")

    model.train()
    for epoch in range(EMBED_EPOCHS):
        perm = torch.randperm(n_samples)
        epoch_loss = 0.0
        n_batches  = 0
        for i in range(0, n_samples, batch_size):
            idx    = perm[i:i + batch_size]
            xb, yb = X_tr_fold[idx], y_tr_fold[idx]
            optimizer.zero_grad()
            loss   = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            n_batches  += 1

        is_last = epoch == EMBED_EPOCHS - 1
        if epoch == 0 or is_last or (epoch + 1) % EMBED_LOG_EVERY == 0:
            elapsed = time.time() - fold_t0
            print(f"    Epoch {epoch + 1:>3}/{EMBED_EPOCHS}  |  "
                  f"loss={epoch_loss / n_batches:.4f}  |  {elapsed:.1f}s elapsed")

    model.eval()
    with torch.no_grad():
        oof_embeddings[ho_idx] = model.embed(X_ho_fold).cpu().numpy()
        oof_probs[ho_idx]      = torch.sigmoid(model(X_ho_fold)).cpu().numpy()

    fold_time = time.time() - fold_t0
    fold_times.append(fold_time)
    remaining_folds = N_FOLDS - fold
    eta = np.mean(fold_times) * remaining_folds

    print(f"  Fold {fold}/{N_FOLDS} done in {fold_time:.1f}s -- {len(ho_idx)} held-out samples "
          f"({int(y_train[ho_idx].sum())} SEP, {len(ho_idx) - int(y_train[ho_idx].sum())} Non-SEP)")
    if remaining_folds > 0:
        print(f"    ETA: ~{eta:.0f}s remaining ({remaining_folds} fold(s) left)")

    # Free this fold's model/tensors before the next fold. Without this,
    # 5 folds' worth of MPS-device models and optimizer state accumulate
    # across the loop -- on Apple Silicon this can exhaust GPU memory and
    # crash the whole kernel (not just raise a catchable Python error),
    # sometimes only manifesting a cell or two later.
    del model, optimizer, criterion, X_tr_fold, y_tr_fold, X_ho_fold, pos_weight
    if device.type == "mps":
        torch.mps.empty_cache()

print("\nOut-of-fold probabilities and embeddings ready.")

Using device: mps
══════════════════════════════════════════════════════════════════════
  Shared out-of-fold GRU -- probabilities + embeddings
  Folds       : 5
  Params      : {'units': 64, 'layers': 1, 'dropout': 0.1, 'lr': 0.001, 'batch_size': 64}
  Epochs/fold : 30 (fixed, no early stopping)
══════════════════════════════════════════════════════════════════════

  Fold 1/5 -- training on 9964 samples (95 SEP, 9869 Non-SEP)...
    Epoch   1/30  |  loss=1.1291  |  13.2s elapsed
    Epoch   5/30  |  loss=0.9550  |  62.4s elapsed
    Epoch  10/30  |  loss=0.8315  |  124.5s elapsed
    Epoch  15/30  |  loss=0.7392  |  187.8s elapsed
    Epoch  20/30  |  loss=0.6690  |  253.3s elapsed
    Epoch  25/30  |  loss=0.5973  |  317.7s elapsed
    Epoch  30/30  |  loss=0.5688  |  383.1s elapsed
  Fold 1/5 done in 383.4s -- 2491 held-out samples (23 SEP, 2468 Non-SEP)
    ETA: ~1534s remaining (4 fold(s) left)

  Fold 2/5 -- training on 9964 samples (95 SEP, 9869 Non-SEP)...
    Epoch   1/30  | 

## 3. Method 1 — Deep ENN (Classifier Confidence)

In [8]:
# ══════════════════════════════════════════════════════════════
# METHOD 1: DEEP ENN -- classifier-confidence-based cleaning
#
# Removes Non-SEP training samples the GRU confidently mistakes for SEP
# (out-of-fold predicted probability >= 0.5) -- a direct deep-learning
# analogue of classical Edited Nearest Neighbours, using a trained
# classifier's confidence instead of a k-NN vote.
# ══════════════════════════════════════════════════════════════

DEEP_ENN_THRESHOLD = 0.95

nonsep_idx    = np.where(y_train == 0)[0]
confused_mask = oof_probs[nonsep_idx] >= DEEP_ENN_THRESHOLD
deep_enn_removed_idx = nonsep_idx[confused_mask]

print(f"Deep ENN (threshold={DEEP_ENN_THRESHOLD}): "
      f"{len(deep_enn_removed_idx)} Non-SEP training samples confidently "
      "scored as SEP-like by the out-of-fold GRU")

keep_mask = np.ones(len(y_train), dtype=bool)
keep_mask[deep_enn_removed_idx] = False

X_train_deep_enn = X_train[keep_mask]
y_train_deep_enn = y_train[keep_mask]

new_counts = np.bincount(y_train_deep_enn.astype(int), minlength=2)
print("Before: Non-SEP:", train_counts[0], " SEP:", train_counts[1])
print("After : Non-SEP:", new_counts[0], " SEP:", new_counts[1])
print("Removed:", len(y_train) - len(y_train_deep_enn))

deep_enn_dir = "./final_split_data_HybridNorm_BestFeatures_DeepENN"
os.makedirs(deep_enn_dir, exist_ok=True)

save_bundle(X_train_deep_enn, y_train_deep_enn, os.path.join(deep_enn_dir, "train_set.pkl"))
save_bundle(X_val,            y_val,            os.path.join(deep_enn_dir, "val_set.pkl"))
save_bundle(X_test,           y_test,           os.path.join(deep_enn_dir, "test_set.pkl"))

print(f"\n✅ Deep ENN dataset saved -> {os.path.abspath(deep_enn_dir)}/")

Deep ENN (threshold=0.95): 72 Non-SEP training samples confidently scored as SEP-like by the out-of-fold GRU
Before: Non-SEP: 12337  SEP: 118
After : Non-SEP: 12265  SEP: 118
Removed: 72
Saved: ./final_split_data_HybridNorm_BestFeatures_DeepENN/train_set.pkl
Saved: ./final_split_data_HybridNorm_BestFeatures_DeepENN/val_set.pkl
Saved: ./final_split_data_HybridNorm_BestFeatures_DeepENN/test_set.pkl

✅ Deep ENN dataset saved -> ./final_split_data_HybridNorm_BestFeatures_DeepENN/


## 4. Method 2 — Deep Tomek Links (Embedding Mutual Nearest-Neighbor)

In [9]:
# ══════════════════════════════════════════════════════════════
# METHOD 2: DEEP TOMEK LINKS -- embedding mutual nearest-neighbor
#
# Classical TomekLinks finds majority/minority mutual-nearest-neighbor
# pairs in raw, flattened Euclidean space -- dominated by waveform
# shape/timing rather than what separates SEP from Non-SEP. This finds
# them instead in the GRU's learned, discriminative embedding (from the
# shared out-of-fold pass in Section 2).
# ══════════════════════════════════════════════════════════════

from sklearn.neighbors import NearestNeighbors

class0_idx = np.where(y_train == 0)[0]
class1_idx = np.where(y_train == 1)[0]

emb0 = oof_embeddings[class0_idx]
emb1 = oof_embeddings[class1_idx]

nn_to_1 = NearestNeighbors(n_neighbors=1).fit(emb1)
_, nearest_1_of_0 = nn_to_1.kneighbors(emb0)   # each class-0 point's nearest class-1 point

nn_to_0 = NearestNeighbors(n_neighbors=1).fit(emb0)
_, nearest_0_of_1 = nn_to_0.kneighbors(emb1)   # each class-1 point's nearest class-0 point

deep_tomek_removed_idx = []
for i0, (j1,) in enumerate(nearest_1_of_0):
    j0_back = nearest_0_of_1[j1][0]
    if j0_back == i0:
        deep_tomek_removed_idx.append(class0_idx[i0])

deep_tomek_removed_idx = np.array(sorted(set(deep_tomek_removed_idx)))

print(f"Deep Tomek Links found: {len(deep_tomek_removed_idx)} Non-SEP training "
      "samples mutually nearest to a SEP sample in the learned embedding space")

keep_mask = np.ones(len(y_train), dtype=bool)
keep_mask[deep_tomek_removed_idx] = False

X_train_deep_tomek = X_train[keep_mask]
y_train_deep_tomek = y_train[keep_mask]

new_counts = np.bincount(y_train_deep_tomek.astype(int), minlength=2)
print("Before: Non-SEP:", train_counts[0], " SEP:", train_counts[1])
print("After : Non-SEP:", new_counts[0], " SEP:", new_counts[1])
print("Removed:", len(y_train) - len(y_train_deep_tomek))

deep_tomek_dir = "./final_split_data_HybridNorm_BestFeatures_DeepTomek"
os.makedirs(deep_tomek_dir, exist_ok=True)

save_bundle(X_train_deep_tomek, y_train_deep_tomek, os.path.join(deep_tomek_dir, "train_set.pkl"))
save_bundle(X_val,              y_val,              os.path.join(deep_tomek_dir, "val_set.pkl"))
save_bundle(X_test,             y_test,             os.path.join(deep_tomek_dir, "test_set.pkl"))

print(f"\n✅ Deep Tomek Links dataset saved -> {os.path.abspath(deep_tomek_dir)}/")

Deep Tomek Links found: 100 Non-SEP training samples mutually nearest to a SEP sample in the learned embedding space
Before: Non-SEP: 12337  SEP: 118
After : Non-SEP: 12237  SEP: 118
Removed: 100
Saved: ./final_split_data_HybridNorm_BestFeatures_DeepTomek/train_set.pkl
Saved: ./final_split_data_HybridNorm_BestFeatures_DeepTomek/val_set.pkl
Saved: ./final_split_data_HybridNorm_BestFeatures_DeepTomek/test_set.pkl

✅ Deep Tomek Links dataset saved -> ./final_split_data_HybridNorm_BestFeatures_DeepTomek/


## 5. Method 3 — Autoencoder Novelty Detection (Trained on SEP Only)

In [15]:
# ══════════════════════════════════════════════════════════════
# METHOD 3: AUTOENCODER NOVELTY DETECTION -- trained on SEP only
#
# Trains a small autoencoder on *only* the SEP training samples. A
# Non-SEP sample the autoencoder reconstructs well lies close to the
# manifold of SEP events it learned -- i.e. it looks like a SEP event
# to a model that has only ever seen SEP events.
#
# Uses a 1D convolutional autoencoder, not a flattened dense MLP: convs
# respect the sequence's time structure (unlike flattening 288 x 5 into
# an unordered 1440-dim vector), and because conv kernels are shared
# across time they also have far fewer parameters (~39k here vs. ~373k
# for the flat MLP version this replaced) -- both a better inductive
# bias *and* less overfitting risk for only ~100 SEP training examples.
#
# Threshold calibration is out-of-fold, same discipline as Methods 1,
# 2, and 4: a single autoencoder trained and then scored on the *same*
# SEP samples nearly memorizes them, making any threshold calibrated
# from it too permissive. K-fold CV *within* the SEP samples fixes
# this: each fold's autoencoder is scored only on SEP samples it never
# trained on, giving a fair error distribution to threshold from, and
# an ensemble of fold-models to score the Non-SEP samples with.
# ══════════════════════════════════════════════════════════════

from sklearn.model_selection import KFold

class ConvAutoEncoder(nn.Module):
    """Encoder: 3x (Conv1d, stride 2) over the time axis -> global average
    pool -> latent vector. Decoder: project back to a short sequence, then
    3x ConvTranspose1d (stride 2) to upsample back to the original length.
    Requires seq_len divisible by 8 (three stride-2 halvings) -- true here
    (288 timesteps)."""

    def __init__(self, n_features, seq_len, latent_dim=16, dropout=0.2):
        super().__init__()
        assert seq_len % 8 == 0, "seq_len must be divisible by 8 for this architecture"
        self.latent_len = seq_len // 8

        self.enc_conv1 = nn.Conv1d(n_features, 16, kernel_size=7, stride=2, padding=3)
        self.enc_conv2 = nn.Conv1d(16, 32, kernel_size=7, stride=2, padding=3)
        self.enc_conv3 = nn.Conv1d(32, latent_dim, kernel_size=7, stride=2, padding=3)
        self.pool      = nn.AdaptiveAvgPool1d(1)
        self.dropout   = nn.Dropout(dropout)

        self.fc_expand = nn.Linear(latent_dim, 32 * self.latent_len)
        self.dec_conv1 = nn.ConvTranspose1d(32, 32, kernel_size=7, stride=2, padding=3, output_padding=1)
        self.dec_conv2 = nn.ConvTranspose1d(32, 16, kernel_size=7, stride=2, padding=3, output_padding=1)
        self.dec_conv3 = nn.ConvTranspose1d(16, n_features, kernel_size=7, stride=2, padding=3, output_padding=1)
        self.relu = nn.ReLU()

    def encode(self, x):
        # x: (batch, T, F) -> (batch, F, T) for Conv1d
        x = x.transpose(1, 2)
        x = self.relu(self.enc_conv1(x))
        x = self.relu(self.enc_conv2(x))
        x = self.relu(self.enc_conv3(x))
        x = self.dropout(x)
        return self.pool(x).squeeze(-1)   # (batch, latent_dim)

    def decode(self, z):
        x = self.fc_expand(z).view(-1, 32, self.latent_len)
        x = self.relu(self.dec_conv1(x))
        x = self.relu(self.dec_conv2(x))
        x = self.dec_conv3(x)             # (batch, F, T)
        return x.transpose(1, 2)          # (batch, T, F)

    def forward(self, x):
        return self.decode(self.encode(x))


def train_ae(X_tensor, epochs, lr, weight_decay, latent_dim):
    ae = ConvAutoEncoder(X_tensor.shape[2], X_tensor.shape[1], latent_dim=latent_dim).to(device)
    optimizer = torch.optim.Adam(ae.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.MSELoss()

    ae.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        loss = criterion(ae(X_tensor), X_tensor)
        loss.backward()
        optimizer.step()

    return ae


AE_FOLDS        = 5
AE_EPOCHS       = 300
AE_LR           = 1e-3
AE_WEIGHT_DECAY = 1e-4
AE_LATENT_DIM   = 64
AE_PERCENTILE   = 0.01   # see note below -- LOW percentile = strict look-alike bar

sep_idx    = np.where(y_train == 1)[0]
nonsep_idx = np.where(y_train == 0)[0]

N, T, Fdim = X_train.shape

X_nonsep_tensor = torch.tensor(X_train[nonsep_idx], dtype=torch.float32).to(device)

print("═" * 70)
print("  Autoencoder novelty detection -- out-of-fold threshold calibration")
print(f"  Architecture: 1D conv autoencoder ({sum(p.numel() for p in ConvAutoEncoder(Fdim, T, AE_LATENT_DIM).parameters()):,} params)")
print(f"  SEP folds   : {AE_FOLDS}")
print(f"  Params      : epochs={AE_EPOCHS}, lr={AE_LR}, weight_decay={AE_WEIGHT_DECAY}, latent_dim={AE_LATENT_DIM}")
print("═" * 70)

ae_kfold = KFold(n_splits=AE_FOLDS, shuffle=True, random_state=42)

oof_sep_error         = np.zeros(len(sep_idx), dtype=np.float32)
nonsep_error_per_fold = np.zeros((AE_FOLDS, len(nonsep_idx)), dtype=np.float32)

for fold, (tr_idx, ho_idx) in enumerate(ae_kfold.split(sep_idx), start=1):

    fold_t0 = time.time()

    X_sep_tr = torch.tensor(X_train[sep_idx[tr_idx]], dtype=torch.float32).to(device)
    X_sep_ho = torch.tensor(X_train[sep_idx[ho_idx]], dtype=torch.float32).to(device)

    ae = train_ae(X_sep_tr, AE_EPOCHS, AE_LR, AE_WEIGHT_DECAY, AE_LATENT_DIM)

    ae.eval()
    with torch.no_grad():
        ho_recon = ae(X_sep_ho)
        oof_sep_error[ho_idx] = ((ho_recon - X_sep_ho) ** 2).mean(dim=(1, 2)).cpu().numpy()

        nonsep_recon = ae(X_nonsep_tensor)
        nonsep_error_per_fold[fold - 1] = ((nonsep_recon - X_nonsep_tensor) ** 2).mean(dim=(1, 2)).cpu().numpy()

    print(f"  Fold {fold}/{AE_FOLDS} done in {time.time() - fold_t0:.1f}s -- "
          f"{len(tr_idx)} SEP train, {len(ho_idx)} SEP held out")

    del ae, X_sep_tr, X_sep_ho
    if device.type == "mps":
        torch.mps.empty_cache()

# Ensemble Non-SEP score: average reconstruction error across the 5
# fold-models, so no single fold's overfitting quirk dominates.
nonsep_error = nonsep_error_per_fold.mean(axis=0)

# AE_PERCENTILE is a percentile of the *SEP* error distribution -- a LOW
# value (e.g. 10) means "reconstructs as well as a well-reconstructed SEP
# sample" (strict, closer to a true look-alike); a HIGH value (e.g. 90)
# means "reconstructs as well as even a poorly-reconstructed SEP sample"
# (loose -- an earlier version of this cell used 90 and flagged ~99% of
# the majority class, because that bar is easy to clear). Printing the
# removal count across several percentiles here instead of silently
# picking one, since the right cutoff is a judgment call.
print("\nSensitivity of removal count to the percentile choice:")
print(f"{'Percentile':>10} {'Threshold':>12} {'Non-SEP removed':>18} {'% of majority':>15}")
for pct in [5, 10, 25, 50, 75, 90]:
    pct_threshold = np.percentile(oof_sep_error, pct)
    pct_removed   = int((nonsep_error <= pct_threshold).sum())
    print(f"{pct:>10} {pct_threshold:>12.6f} {pct_removed:>18} {100 * pct_removed / len(nonsep_idx):>14.1f}%")

threshold = np.percentile(oof_sep_error, AE_PERCENTILE)

print(f"\nOut-of-fold SEP reconstruction error: min={oof_sep_error.min():.6f} "
      f"median={np.median(oof_sep_error):.6f} max={oof_sep_error.max():.6f}")
print(f"Using threshold (out-of-fold SEP {AE_PERCENTILE}th percentile): {threshold:.6f}")

confused_mask = nonsep_error <= threshold
deep_ae_removed_idx = nonsep_idx[confused_mask]

print(f"\nAutoencoder novelty detection: {len(deep_ae_removed_idx)} Non-SEP "
      "training samples reconstruct (ensemble-averaged) as well as a "
      f"well-reconstructed (top {AE_PERCENTILE}th percentile) held-out SEP sample")

keep_mask = np.ones(len(y_train), dtype=bool)
keep_mask[deep_ae_removed_idx] = False

X_train_deep_ae = X_train[keep_mask]
y_train_deep_ae = y_train[keep_mask]

new_counts = np.bincount(y_train_deep_ae.astype(int), minlength=2)
print("Before: Non-SEP:", train_counts[0], " SEP:", train_counts[1])
print("After : Non-SEP:", new_counts[0], " SEP:", new_counts[1])
print("Removed:", len(y_train) - len(y_train_deep_ae))

deep_ae_dir = "./final_split_data_HybridNorm_BestFeatures_DeepAE"
os.makedirs(deep_ae_dir, exist_ok=True)

save_bundle(X_train_deep_ae, y_train_deep_ae, os.path.join(deep_ae_dir, "train_set.pkl"))
save_bundle(X_val,           y_val,           os.path.join(deep_ae_dir, "val_set.pkl"))
save_bundle(X_test,          y_test,          os.path.join(deep_ae_dir, "test_set.pkl"))

print(f"\n✅ Autoencoder-cleaned dataset saved -> {os.path.abspath(deep_ae_dir)}/")

══════════════════════════════════════════════════════════════════════
  Autoencoder novelty detection -- out-of-fold threshold calibration
  Architecture: 1D conv autoencoder (104,387 params)
  SEP folds   : 5
  Params      : epochs=300, lr=0.001, weight_decay=0.0001, latent_dim=64
══════════════════════════════════════════════════════════════════════
  Fold 1/5 done in 1.6s -- 94 SEP train, 24 SEP held out
  Fold 2/5 done in 1.4s -- 94 SEP train, 24 SEP held out
  Fold 3/5 done in 1.4s -- 94 SEP train, 24 SEP held out
  Fold 4/5 done in 1.4s -- 95 SEP train, 23 SEP held out
  Fold 5/5 done in 1.4s -- 95 SEP train, 23 SEP held out

Sensitivity of removal count to the percentile choice:
Percentile    Threshold    Non-SEP removed   % of majority
         5     0.106105               2314           18.8%
        10     0.126146               3463           28.1%
        25     0.177920               5941           48.2%
        50     0.291172               9227           74.8%
        7

## 6. Method 4 — Siamese/Triplet Network (Learned Similarity)

In [26]:
# ══════════════════════════════════════════════════════════════
# METHOD 4: SIAMESE / TRIPLET NETWORK -- learned similarity metric
#
# Unlike Deep Tomek Links (which borrows a classifier's hidden layer as
# a byproduct), this trains an embedding *explicitly* for similarity:
# triplet loss directly optimizes same-class pairs to be close and
# cross-class pairs to be far apart. Trained out-of-fold (5-fold, same
# splits as Section 2) so a sample's embedding never comes from a model
# that trained directly on it.
#
# Non-SEP samples still close to a SEP sample in this space *despite*
# training that explicitly pushes the classes apart are the most
# stubbornly ambiguous points of all four methods here.
# ══════════════════════════════════════════════════════════════

from sklearn.neighbors import NearestNeighbors


class TripletEncoder(nn.Module):
    def __init__(self, input_size, hidden_size=32, embed_dim=16, dropout=0.1):
        super().__init__()
        self.gru  = nn.GRU(input_size, hidden_size, batch_first=True)
        self.drop = nn.Dropout(dropout)
        self.fc   = nn.Linear(hidden_size, embed_dim)

    def forward(self, x):
        out, _ = self.gru(x)
        out    = out[:, -1, :]
        out    = self.drop(out)
        out    = self.fc(out)
        return nn.functional.normalize(out, p=2, dim=1)  # unit hypersphere


def sample_triplets(y, n_triplets, rng):
    pos_idx = np.where(y == 1)[0]
    neg_idx = np.where(y == 0)[0]
    triplets = []
    for _ in range(n_triplets):
        anchor_is_pos = rng.integers(0, 2) == 1
        same_pool  = pos_idx if anchor_is_pos else neg_idx
        other_pool = neg_idx if anchor_is_pos else pos_idx
        if len(same_pool) < 2 or len(other_pool) < 1:
            continue
        a, p = rng.choice(same_pool, size=2, replace=False)
        n = rng.choice(other_pool)
        triplets.append((a, p, n))
    return np.array(triplets)


TRIPLET_EPOCHS     = 30
TRIPLET_LOG_EVERY  = 5   # print progress every N epochs, plus first/last
TRIPLETS_PER_EPOCH = 256
TRIPLET_BATCH_SIZE = 64
TRIPLET_MARGIN     = 1.0
SIAMESE_PERCENTILE = 5   # see note below -- LOW percentile = strict look-alike bar
triplet_params = {"hidden_size": 64, "embed_dim": 32, "dropout": 0.1, "lr": 0.001}

rng = np.random.default_rng(42)
oof_triplet_embeddings = np.zeros((len(y_train), triplet_params["embed_dim"]), dtype=np.float32)

print("═" * 70)
print("  Siamese/Triplet network -- out-of-fold embedding")
print(f"  Folds       : {N_FOLDS}")
print(f"  Params      : {triplet_params}")
print(f"  Epochs/fold : {TRIPLET_EPOCHS}, {TRIPLETS_PER_EPOCH} triplets/epoch")
print("═" * 70)

fold_times = []

for fold, (tr_idx, ho_idx) in enumerate(skf.split(X_train, y_train), start=1):

    fold_t0 = time.time()

    X_tr_fold = X_train[tr_idx]
    y_tr_fold = y_train[tr_idx]

    encoder = TripletEncoder(X_train.shape[2],
                              hidden_size=triplet_params["hidden_size"],
                              embed_dim=triplet_params["embed_dim"],
                              dropout=triplet_params["dropout"]).to(device)

    triplet_loss = nn.TripletMarginLoss(margin=TRIPLET_MARGIN)
    optimizer    = torch.optim.Adam(encoder.parameters(), lr=triplet_params["lr"])

    print(f"\n  Fold {fold}/{N_FOLDS} -- training on {len(tr_idx)} samples "
          f"({int(y_tr_fold.sum())} SEP, {len(tr_idx) - int(y_tr_fold.sum())} Non-SEP)...")

    encoder.train()
    for epoch in range(TRIPLET_EPOCHS):
        triplets = sample_triplets(y_tr_fold, TRIPLETS_PER_EPOCH, rng)
        if len(triplets) == 0:
            continue

        epoch_loss = 0.0
        n_batches  = 0
        for i in range(0, len(triplets), TRIPLET_BATCH_SIZE):
            batch = triplets[i:i + TRIPLET_BATCH_SIZE]
            a_idx, p_idx, n_idx = batch[:, 0], batch[:, 1], batch[:, 2]

            Xa = torch.tensor(X_tr_fold[a_idx], dtype=torch.float32).to(device)
            Xp = torch.tensor(X_tr_fold[p_idx], dtype=torch.float32).to(device)
            Xn = torch.tensor(X_tr_fold[n_idx], dtype=torch.float32).to(device)

            optimizer.zero_grad()
            loss = triplet_loss(encoder(Xa), encoder(Xp), encoder(Xn))
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            n_batches  += 1

        is_last = epoch == TRIPLET_EPOCHS - 1
        if n_batches > 0 and (epoch == 0 or is_last or (epoch + 1) % TRIPLET_LOG_EVERY == 0):
            elapsed = time.time() - fold_t0
            print(f"    Epoch {epoch + 1:>3}/{TRIPLET_EPOCHS}  |  "
                  f"loss={epoch_loss / n_batches:.4f}  |  {elapsed:.1f}s elapsed")

    encoder.eval()
    with torch.no_grad():
        X_ho_fold = torch.tensor(X_train[ho_idx], dtype=torch.float32).to(device)
        oof_triplet_embeddings[ho_idx] = encoder(X_ho_fold).cpu().numpy()

    fold_time = time.time() - fold_t0
    fold_times.append(fold_time)
    remaining_folds = N_FOLDS - fold
    eta = np.mean(fold_times) * remaining_folds

    print(f"  Fold {fold}/{N_FOLDS} done in {fold_time:.1f}s -- {len(ho_idx)} held-out samples embedded")
    if remaining_folds > 0:
        print(f"    ETA: ~{eta:.0f}s remaining ({remaining_folds} fold(s) left)")

    # Same reasoning as Section 2 -- free this fold's model/tensors before
    # starting the next one to avoid MPS memory buildup across folds.
    del encoder, optimizer, X_ho_fold
    if device.type == "mps":
        torch.mps.empty_cache()

print(f"\nOut-of-fold triplet embeddings ready: {oof_triplet_embeddings.shape}")

# ---------------------------------------------------------
# Removal rule: Non-SEP samples whose nearest SEP neighbor is at least
# as close as SEP samples typically are to each other -- a
# self-calibrated threshold using the SEP-to-SEP nearest-neighbor
# distance scale from this same learned space.
# ---------------------------------------------------------
# SIAMESE_PERCENTILE is a percentile of the *SEP-to-SEP* nearest-neighbor
# distance distribution -- a LOW value (e.g. 10) means "within the
# distance of the tightest, closest-clustered SEP pairs" (strict); a HIGH
# value (e.g. 75) means "within the distance of even loosely-clustered
# SEP pairs" (loose -- it has to be a wide radius to capture 75% of
# same-class pairs, which is why SIAMESE_PERCENTILE=75 flagged 57% of the
# majority class). Same directional bug as the autoencoder's threshold
# before that got fixed -- printing the full sensitivity table instead of
# picking a percentile silently.
# ---------------------------------------------------------
class0_idx = np.where(y_train == 0)[0]
class1_idx = np.where(y_train == 1)[0]

emb0 = oof_triplet_embeddings[class0_idx]
emb1 = oof_triplet_embeddings[class1_idx]

nn_sep_sep = NearestNeighbors(n_neighbors=2).fit(emb1)
sep_sep_dist, _ = nn_sep_sep.kneighbors(emb1)
sep_sep_nn_dist = sep_sep_dist[:, 1]  # column 0 is distance to self (0.0)

nn_to_1 = NearestNeighbors(n_neighbors=1).fit(emb1)
nonsep_to_sep_dist, _ = nn_to_1.kneighbors(emb0)
nonsep_to_sep_dist = nonsep_to_sep_dist[:, 0]

print("\nSensitivity of removal count to the percentile choice:")
print(f"{'Percentile':>10} {'Threshold':>12} {'Non-SEP removed':>18} {'% of majority':>15}")
for pct in [5, 10, 25, 50, 75, 90]:
    pct_threshold = np.percentile(sep_sep_nn_dist, pct)
    pct_removed   = int((nonsep_to_sep_dist <= pct_threshold).sum())
    print(f"{pct:>10} {pct_threshold:>12.4f} {pct_removed:>18} {100 * pct_removed / len(class0_idx):>14.1f}%")

distance_threshold = np.percentile(sep_sep_nn_dist, SIAMESE_PERCENTILE)

print(f"\nSEP-to-SEP nearest-neighbor distance: median={np.median(sep_sep_nn_dist):.4f}  "
      f"using {SIAMESE_PERCENTILE}th pct={distance_threshold:.4f}")

confused_mask = nonsep_to_sep_dist <= distance_threshold
siamese_removed_idx = class0_idx[confused_mask]

print(f"Siamese/Triplet network: {len(siamese_removed_idx)} Non-SEP training "
      f"samples within the {SIAMESE_PERCENTILE}th-percentile SEP-to-SEP distance of a SEP sample")

keep_mask = np.ones(len(y_train), dtype=bool)
keep_mask[siamese_removed_idx] = False

X_train_siamese = X_train[keep_mask]
y_train_siamese = y_train[keep_mask]

new_counts = np.bincount(y_train_siamese.astype(int), minlength=2)
print("Before: Non-SEP:", train_counts[0], " SEP:", train_counts[1])
print("After : Non-SEP:", new_counts[0], " SEP:", new_counts[1])
print("Removed:", len(y_train) - len(y_train_siamese))

siamese_dir = "./final_split_data_HybridNorm_BestFeatures_Siamese"
os.makedirs(siamese_dir, exist_ok=True)

save_bundle(X_train_siamese, y_train_siamese, os.path.join(siamese_dir, "train_set.pkl"))
save_bundle(X_val,           y_val,           os.path.join(siamese_dir, "val_set.pkl"))
save_bundle(X_test,          y_test,          os.path.join(siamese_dir, "test_set.pkl"))

print(f"\n✅ Siamese/Triplet-cleaned dataset saved -> {os.path.abspath(siamese_dir)}/")

══════════════════════════════════════════════════════════════════════
  Siamese/Triplet network -- out-of-fold embedding
  Folds       : 5
  Params      : {'hidden_size': 64, 'embed_dim': 32, 'dropout': 0.1, 'lr': 0.001}
  Epochs/fold : 30, 256 triplets/epoch
══════════════════════════════════════════════════════════════════════

  Fold 1/5 -- training on 9964 samples (95 SEP, 9869 Non-SEP)...
    Epoch   1/30  |  loss=0.8357  |  1.2s elapsed
    Epoch   5/30  |  loss=0.7161  |  5.1s elapsed
    Epoch  10/30  |  loss=0.6128  |  10.0s elapsed
    Epoch  15/30  |  loss=0.5816  |  14.9s elapsed
    Epoch  20/30  |  loss=0.7393  |  19.7s elapsed
    Epoch  25/30  |  loss=0.5855  |  24.6s elapsed
    Epoch  30/30  |  loss=0.6337  |  29.5s elapsed
  Fold 1/5 done in 29.6s -- 2491 held-out samples embedded
    ETA: ~118s remaining (4 fold(s) left)

  Fold 2/5 -- training on 9964 samples (95 SEP, 9869 Non-SEP)...
    Epoch   1/30  |  loss=0.9043  |  1.0s elapsed
    Epoch   5/30  |  loss=0.70

## 7. Summary

In [27]:
summary_rows = [
    ("Deep ENN",          np.bincount(y_train_deep_enn.astype(int), minlength=2),   deep_enn_dir),
    ("Deep Tomek Links",  np.bincount(y_train_deep_tomek.astype(int), minlength=2), deep_tomek_dir),
    ("Autoencoder (SEP)", np.bincount(y_train_deep_ae.astype(int), minlength=2),    deep_ae_dir),
    ("Siamese/Triplet",   np.bincount(y_train_siamese.astype(int), minlength=2),    siamese_dir),
]

print("═" * 100)
print("  Summary -- 4 neural-network-based borderline cleaning methods")
print("═" * 100)
print(f"{'Method':<20} {'Non-SEP before':>15} {'Non-SEP after':>15} {'Removed':>10} {'SEP kept':>10}  Output")
print("─" * 100)
for name, after, out_dir in summary_rows:
    removed = train_counts[0] - after[0]
    print(f"{name:<20} {train_counts[0]:>15} {after[0]:>15} {removed:>10} {after[1]:>10}  {out_dir}")

print("\nNext: notebook 4 trains a GRU (5 runs each) on all four of these "
      "datasets to see which cleaning method actually improves TSS.")

════════════════════════════════════════════════════════════════════════════════════════════════════
  Summary -- 4 neural-network-based borderline cleaning methods
════════════════════════════════════════════════════════════════════════════════════════════════════
Method                Non-SEP before   Non-SEP after    Removed   SEP kept  Output
────────────────────────────────────────────────────────────────────────────────────────────────────
Deep ENN                       12337           12265         72        118  ./final_split_data_HybridNorm_BestFeatures_DeepENN
Deep Tomek Links               12337           12237        100        118  ./final_split_data_HybridNorm_BestFeatures_DeepTomek
Autoencoder (SEP)              12337           11943        394        118  ./final_split_data_HybridNorm_BestFeatures_DeepAE
Siamese/Triplet                12337           12262         75        118  ./final_split_data_HybridNorm_BestFeatures_Siamese

Next: notebook 4 trains a GRU (5 runs ea